# 🎯 Milestone 3: RAG + Semantic Search Layer

**AI Knowledge Graph Builder for Enterprise Intelligence**

---

## 📋 What This Notebook Does

Builds a **LangChain + FAISS + Groq** RAG pipeline on top of the Neo4j Knowledge Graph from Milestone 2.

| Component | Tool | Cost |
|---|---|---|
| Vector Store | FAISS | ✅ Free |
| Embeddings | SentenceTransformers | ✅ Free |
| LLM | Groq Llama 3 | ✅ Free |
| RAG Framework | LangChain | ✅ Free |
| Graph DB | Neo4j (Milestone 2) | ✅ Free |


# 📦 PART 1: Setup & Installation

## 1.1 Install Dependencies

In [1]:
# Install all required packages
!pip install langchain langchain-groq langchain-community langchain-core sentence-transformers faiss-cpu neo4j --upgrade -q

print("✅ Installed:")
print("  • langchain          (RAG framework)")
print("  • langchain-groq     (Groq LLM integration)")
print("  • langchain-community (FAISS + document loaders)")
print("  • sentence-transformers (local embeddings)")
print("  • faiss-cpu          (vector search)")
print("  • neo4j              (graph database)")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
✅ Installed:
  •

## 1.2 Import Libraries

In [2]:
import json
import numpy as np
from dataclasses import dataclass
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# Neo4j
from neo4j import GraphDatabase

# LangChain - RAG Framework (latest 2024 imports)
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("✅ All libraries imported successfully")
print("  • LangChain RAG framework ready")
print("  • Groq LLM integration ready")
print("  • FAISS vector store ready")

✅ All libraries imported successfully
  • LangChain RAG framework ready
  • Groq LLM integration ready
  • FAISS vector store ready


## 1.3 Configuration

**⚠️ IMPORTANT: Update credentials below!**

In [3]:
# ════════════════════════════════════════════════════════════════
# CONFIGURATION - UPDATE THESE!
# ════════════════════════════════════════════════════════════════

# Neo4j Connection (from Milestone 2)
NEO4J_URI = "neo4j+s://7374fcc9.databases.neo4j.io"  # connection URI
NEO4J_USER = "7374fcc9"                               # Usually "neo4j"
NEO4J_PASSWORD = "wv-6DB6lq8huzpGe3nuCJdQbp1xESRRDughj2EjQ7ng"  # password

# FREE Groq API key - get at console.groq.com
GROQ_API_KEY = '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' #free Groq key

# Embedding Model (local, no API key needed)
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

# LLM Model
LLM_MODEL = 'llama-3.3-70b-versatile'

# Search Parameters
TOP_K_RESULTS = 10

print("✅ Configuration set")
print(f"  • Embedding Model : {EMBEDDING_MODEL}")
print(f"  • LLM             : {LLM_MODEL} via Groq (FREE)")
print(f"  • RAG Framework   : LangChain")
print(f"  • Vector Store    : FAISS")


✅ Configuration set
  • Embedding Model : all-MiniLM-L6-v2
  • LLM             : llama-3.3-70b-versatile via Groq (FREE)
  • RAG Framework   : LangChain
  • Vector Store    : FAISS


---
# 📊 PART 2: Load Data from Neo4j

## 2.1 Define Job Data Structure

In [21]:
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Job:
    job_id: str
    category: str
    workplace: str
    employment_type: str
    priority_class: str
    demand_score: float
    city: str
    country: str
    region: Optional[str] = None
    department_category: Optional[str] = None
    skills_required: List[str] = None

    def __post_init__(self):
        if self.skills_required is None:
            self.skills_required = []

    @property
    def text_description(self) -> str:
        parts = [
            f"Job Category: {self.category}",
            f"Workplace: {self.workplace}",
            f"Employment Type: {self.employment_type}",
            f"Priority: {self.priority_class}",
            f"Demand Score: {self.demand_score:.1f}",
            f"Location: {self.city}, {self.country}",
        ]
        if self.region:
            parts.append(f"Region: {self.region}")
        if self.department_category:
            parts.append(f"Department Category: {self.department_category}")
        if self.skills_required:
            parts.append(f"Required Skills: {', '.join(self.skills_required)}")
        return "\n".join(parts)

print("✅ Job dataclass defined with skills_required")

✅ Job dataclass defined with skills_required


## 2.2 Load Jobs from Neo4j

In [22]:
def load_jobs_from_neo4j(uri, user, password):
    driver = GraphDatabase.driver(uri, auth=(user, password))

    query = """
    MATCH (j:Job)-[:LOCATED_IN]->(l:Location)
    MATCH (j)-[:BELONGS_TO]->(c:Category)
    OPTIONAL MATCH (j)-[:IN_DEPARTMENT]->(d:Department)
    OPTIONAL MATCH (j)-[:REQUIRES]->(s:Skill)
    RETURN
        j.id                        AS job_id,
        c.name                      AS category,
        j.workplace                 AS workplace,
        j.employment_type           AS employment_type,
        j.priority_class            AS priority_class,
        toFloat(j.demand_score)     AS demand_score,
        l.city                      AS city,
        l.country                   AS country,
        l.region                    AS region,
        d.category                  AS department_category,
        collect(DISTINCT s.name)    AS skills_required
    """

    jobs = []
    with driver.session() as session:
        results = session.run(query)
        for record in results:
            job = Job(
                job_id              = str(record["job_id"]),
                category            = record["category"],
                workplace           = record["workplace"],
                employment_type     = record["employment_type"],
                priority_class      = record["priority_class"],
                demand_score        = record["demand_score"],
                city                = record["city"] or "Unknown",
                country             = record["country"] or "Unknown",
                region              = record["region"],
                department_category = record["department_category"],
                skills_required     = record["skills_required"] or []
            )
            jobs.append(job)

    driver.close()
    return jobs

# Load jobs
print("📥 Loading jobs from Neo4j...")
try:
    jobs = load_jobs_from_neo4j(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)
    print(f"✅ Loaded {len(jobs)} jobs from Neo4j")
    sample = jobs[0]
    print(f"\n📋 Sample job:")
    print(f"  ID       : {sample.job_id}")
    print(f"  Category : {sample.category}")
    print(f"  Location : {sample.city}, {sample.country}")
    print(f"  Skills   : {sample.skills_required[:3]}")
    print(f"\n📄 Text Description Preview:")
    print(sample.text_description)
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n⚠️  Check your Neo4j credentials in Section 1.3!")

📥 Loading jobs from Neo4j...
✅ Loaded 644 jobs from Neo4j

📋 Sample job:
  ID       : JOB_000001
  Category : Business Analyst
  Location : Unknown, United Kingdom
  Skills   : ['Excel', 'SQL', 'Process Mapping']

📄 Text Description Preview:
Job Category: Business Analyst
Workplace: Remote
Employment Type: Full-Time
Priority: Normal-High
Demand Score: 26.9
Location: Unknown, United Kingdom
Region: Europe
Department Category: Operations
Required Skills: Excel, SQL, Process Mapping


---
# 🔢 PART 3: Build LangChain + FAISS Vector Store

## 3.1 Convert Jobs to LangChain Documents

In [24]:
print("📄 Converting jobs to LangChain Documents...")

documents = []
for job in jobs:
    doc = Document(
        page_content=job.text_description,
        metadata={
            "job_id"             : job.job_id,
            "category"           : job.category,
            "workplace"          : job.workplace,
            "employment_type"    : job.employment_type,
            "priority_class"     : job.priority_class,
            "demand_score"       : job.demand_score,
            "city"               : job.city,
            "country"            : job.country,
            "region"             : job.region,
            "department_category": job.department_category,
            "skills"             : ", ".join(job.skills_required) if job.skills_required else ""
        }
    )
    documents.append(doc)

print(f"✅ Created {len(documents)} LangChain Documents")
print(f"   Each document has text content + metadata including skills")

📄 Converting jobs to LangChain Documents...
✅ Created 644 LangChain Documents
   Each document has text content + metadata including skills


## 3.2 Load Embedding Model

In [25]:
print("📥 Loading HuggingFace embedding model...")
print(f"  Model: {EMBEDDING_MODEL}")
print("  This may take 1-2 minutes on first run...\n")

# HuggingFaceEmbeddings - LangChain compatible, runs locally, FREE
embeddings_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print("✅ Embedding model loaded!")
print("  • Runs locally — no API key needed")
print("  • Converts text to 384-dimensional vectors")


📥 Loading HuggingFace embedding model...
  Model: all-MiniLM-L6-v2
  This may take 1-2 minutes on first run...



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded!
  • Runs locally — no API key needed
  • Converts text to 384-dimensional vectors


## 3.3 Build FAISS Vector Store with LangChain

In [26]:
print("📊 Building LangChain FAISS Vector Store...")
print(f"  Indexing {len(documents)} job documents...")
print("  This takes ~30 seconds...\n")

# LangChain's FAISS wrapper — handles embeddings + indexing automatically
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings_model
)

# MMR Hybrid Search Retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K_RESULTS, "fetch_k": 30}
)

print(f"✅ FAISS Vector Store built!")
print(f"  • Total vectors : {vectorstore.index.ntotal}")
print(f"  • Index type    : Flat L2 (exact search)")
print(
    f"  • Retriever     : MMR Hybrid Search (fetch 30 → best {TOP_K_RESULTS})")
print(f"  • Ready for RAG pipeline!")

📊 Building LangChain FAISS Vector Store...
  Indexing 644 job documents...
  This takes ~30 seconds...

✅ FAISS Vector Store built!
  • Total vectors : 644
  • Index type    : Flat L2 (exact search)
  • Retriever     : MMR Hybrid Search (fetch 30 → best 10)
  • Ready for RAG pipeline!


---
# 🤖 PART 4: Setup Groq LLM with LangChain

## 4.1 Initialize Groq LLM

In [27]:
# Initialize Groq LLM via LangChain
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name=LLM_MODEL,
    temperature=0.3,
    max_tokens=512
)

print("✅ Groq LLM initialized via LangChain!")
print(f"  • Model    : {LLM_MODEL}")
print(f"  • Provider : Groq (FREE)")
print(f"  • Cost     : $0.00")


✅ Groq LLM initialized via LangChain!
  • Model    : llama-3.3-70b-versatile
  • Provider : Groq (FREE)
  • Cost     : $0.00


## 4.2 Create RAG Prompt Template

In [28]:
# Define the RAG prompt template
# LangChain fills {context} with retrieved job documents
# and {question} with the user's query

RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an intelligent job search assistant for an enterprise knowledge graph system.

Use the following job listings retrieved from the knowledge graph to answer the question.

Retrieved Job Listings:
{context}

User Question: {question}

Instructions:
- Answer based ONLY on the retrieved job listings above
- Mention how many jobs were found
- Highlight key locations, departments, or patterns
- Be concise and helpful (2-4 sentences)
- If no jobs match, say so clearly

Answer:"""
)

print("✅ RAG Prompt Template created")
print("  • Uses retrieved context from FAISS")
print("  • Groq generates answer based on actual job data")


✅ RAG Prompt Template created
  • Uses retrieved context from FAISS
  • Groq generates answer based on actual job data


## 4.3 Build LangChain RAG Chain

In [29]:
# Modern LangChain RAG chain (no RetrievalQA needed)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ LangChain RAG Chain built!")
print()
print("  Pipeline Flow:")
print("  User Query")
print("      ↓")
print("  FAISS retrieves top-10 similar jobs (MMR)")
print("      ↓")
print("  Jobs added as context to prompt")
print("      ↓")
print("  Groq (Llama 3) generates answer")
print("      ↓")
print("  Natural language response returned")

✅ LangChain RAG Chain built!

  Pipeline Flow:
  User Query
      ↓
  FAISS retrieves top-10 similar jobs (MMR)
      ↓
  Jobs added as context to prompt
      ↓
  Groq (Llama 3) generates answer
      ↓
  Natural language response returned


---
# 🔍 PART 5: Semantic Search Function

## 5.1 Define Search Function

In [30]:
def semantic_search(query: str):
    print(f"\n🔍 Query: {query}")
    print("=" * 70)

    print("\n1️⃣  FAISS retrieving similar jobs (MMR)...")
    print("2️⃣  Adding jobs as context to LangChain prompt...")
    print("3️⃣  Groq generating answer...\n")

    # Get retrieved documents
    retrieved_docs = retriever.invoke(query)

    # Run RAG chain
    answer = rag_chain.invoke(query)

    # Extract metadata from retrieved docs
    results = [doc.metadata for doc in retrieved_docs]

    print("=" * 70)
    return answer, results

print("✅ Semantic search function ready!")

✅ Semantic search function ready!


---
# 🧪 PART 6: Test Queries

## 6.1 Query 1: Remote Jobs in India

In [31]:
answer, results = semantic_search("Show me remote full-time jobs in India")

print("\n📝 ANSWER:")
print(answer)

print("\n📊 TOP RETRIEVED JOBS:")
for i, job in enumerate(results[:5], 1):
    print(f"  {i}. {job['category']} | {job['workplace']} | {job['city']}, {job['country']} | {job['priority_class']}")



🔍 Query: Show me remote full-time jobs in India

1️⃣  FAISS retrieving similar jobs (MMR)...
2️⃣  Adding jobs as context to LangChain prompt...
3️⃣  Groq generating answer...


📝 ANSWER:
4 remote full-time jobs were found in India. The jobs are located in various cities including Gurugram, Hyderabad, Delhi, and Unknown, India, with departments ranging from Technology & Data to Marketing & Sales and HR & Management. Key locations include Gurugram and Delhi, with patterns showing a mix of technical and non-technical roles such as Cloud, Data Scientist, and Ui/Ux. The jobs require a range of skills, including Cloud Computing, Python, and HR Analytics.

📊 TOP RETRIEVED JOBS:
  1. Hr | Remote | Unknown, India | Normal-High
  2. Ui/Ux | Remote | Lagos, Nigeria | Premium
  3. Software Developer | Remote | Mandaluyong, Philippines | Premium
  4. Cloud | Remote | Gurugram, India | Normal-High
  5. Data Scientist | Remote | Hyderabad, India | Normal


## 6.2 Query 2: IT Jobs in Bangalore

In [32]:
answer, results = semantic_search("Find IT department jobs in Bangalore")

print("\n📝 ANSWER:")
print(answer)

print("\n📊 TOP RETRIEVED JOBS:")
for i, job in enumerate(results[:5], 1):
    print(f"  {i}. {job['category']} | {job['workplace']} | {job['city']}, {job['country']} | {job['department_category']}")



🔍 Query: Find IT department jobs in Bangalore

1️⃣  FAISS retrieving similar jobs (MMR)...
2️⃣  Adding jobs as context to LangChain prompt...
3️⃣  Groq generating answer...


📝 ANSWER:
3 jobs were found in Bangalore, all of which are in the Asia Pacific region. The jobs are categorized as Cloud, Ui/Ux, and none are specifically in the IT department, but they are related to technology and are listed under the "Other" department category. Key locations include Bangalore, India, and key patterns show that these jobs are mostly hybrid and full-time. These jobs may be of interest to those looking for IT-related roles in Bangalore.

📊 TOP RETRIEVED JOBS:
  1. Data Scientist | On-Site | Bengaluru, India | Technology & Data
  2. Software Developer | Hybrid | Durban, South Africa | Other
  3. Cloud | Hybrid | Bangalore, India | Other
  4. Business Analyst | Remote | Delhi, India | Other
  5. Ui/Ux | Hybrid | Bangalore, India | Other


## 6.3 Query 3: Data Scientist Jobs

In [33]:
answer, results = semantic_search("Data scientist positions with high demand")

print("\n📝 ANSWER:")
print(answer)

print("\n📊 TOP RETRIEVED JOBS:")
for i, job in enumerate(results[:5], 1):
    print(f"  {i}. {job['category']} | {job['workplace']} | {job['city']}, {job['country']} | Demand: {job['demand_score']:.1f}")



🔍 Query: Data scientist positions with high demand

1️⃣  FAISS retrieving similar jobs (MMR)...
2️⃣  Adding jobs as context to LangChain prompt...
3️⃣  Groq generating answer...


📝 ANSWER:
We found 10 data scientist job listings, with 2 positions having a relatively high demand score (above 40). The high-demand jobs are located in unknown locations in the United States (Remote) and Paris, France (Hybrid), belonging to the "Other" and "Technology & Data" departments, respectively. Key locations with multiple data scientist job listings include the United States, India, and the United Kingdom, with required skills often featuring Python, SQL, and Machine Learning.

📊 TOP RETRIEVED JOBS:
  1. Data Scientist | Remote | Unknown, United States | Demand: 46.1
  2. Data Scientist | Hybrid | Paris, France | Demand: 17.3
  3. Data Scientist | On-Site | New Delhi, India | Demand: 3.9
  4. Data Scientist | Unknown | Arlington, United States | Demand: 9.6
  5. Data Scientist | On-Site | San Leand

## 6.4 Query 4: Premium Positions in Europe

In [34]:
answer, results = semantic_search("Premium job positions in Europe")

print("\n📝 ANSWER:")
print(answer)

print("\n📊 TOP RETRIEVED JOBS:")
for i, job in enumerate(results[:5], 1):
    print(f"  {i}. {job['category']} | {job['workplace']} | {job['city']}, {job['country']} | {job['priority_class']}")



🔍 Query: Premium job positions in Europe

1️⃣  FAISS retrieving similar jobs (MMR)...
2️⃣  Adding jobs as context to LangChain prompt...
3️⃣  Groq generating answer...


📝 ANSWER:
Only 1 premium job position was found in Europe, which is a Business Analyst role located in London, United Kingdom, within the Technology & Data department. This job offers a remote workplace and has a high demand score of 100.0. The required skills for this position include Agile, SQL, and Data Modeling, indicating a strong focus on data-driven decision making.

📊 TOP RETRIEVED JOBS:
  1. Hr | Hybrid | Milan, Italy | Normal-High
  2. Cloud | Hybrid | Athens, Greece | Normal-High
  3. Business Analyst | Hybrid | Sofia, Bulgaria | Normal-High
  4. Hr | On-Site | Brussels, Belgium | Normal
  5. Ui/Ux | On-Site | Athens, Greece | Normal


## 6.5 Your Custom Query

Try your own natural language query!

In [35]:
# Try your own query here!
# my_query = "hybrid software developer jobs in UK"
my_query = input("Enter your query: ")

answer, results = semantic_search(my_query)

print("\n📝 ANSWER:")
print(answer)

print(f"\n📊 FOUND {len(results)} JOBS:")
for i, job in enumerate(results[:5], 1):
    print(f"  {i}. {job['category']} | {job['workplace']} | {job['city']}, {job['country']}")


Enter your query: Jobs requiring Python and SQL skills

🔍 Query: Jobs requiring Python and SQL skills

1️⃣  FAISS retrieving similar jobs (MMR)...
2️⃣  Adding jobs as context to LangChain prompt...
3️⃣  Groq generating answer...


📝 ANSWER:
10 jobs were found requiring Python and SQL skills. Key locations for these jobs include the United Kingdom, United States, Greece, Iraq, and the United Arab Emirates, with various departments such as Operations, Technology & Data, and Other. Notably, all job listings require a combination of Python and SQL, often paired with additional skills like Statistical Methods, Cloud Platforms, and Machine Learning. These jobs are scattered across different regions, including Europe, North America, and other areas, with both on-site and remote work options available.

📊 FOUND 10 JOBS:
  1. Data Scientist | Remote | Bristol, United Kingdom
  2. Data Scientist | On-Site | Boulder, United States
  3. Data Scientist | Remote | Unknown, Greece
  4. Data Scientist

---
# 📊 PART 7: System Statistics

In [36]:
print("=" * 70)
print("  MILESTONE 3 - SYSTEM STATISTICS")
print("=" * 70)

print(f"\n📊 Data:")
print(f"  Total jobs indexed : {len(jobs)}")
print(f"  FAISS vectors      : {vectorstore.index.ntotal}")
print(f"  LangChain Documents: {len(documents)}")

print(f"\n🔧 Stack:")
print(f"  RAG Framework : LangChain")
print(f"  Vector Store  : FAISS (Flat L2)")
print(f"  Embeddings    : {EMBEDDING_MODEL}")
print(f"  LLM           : {LLM_MODEL} via Groq")
print(f"  Graph DB      : Neo4j (Milestone 2)")
print(f"  Total Cost    : $0.00 ✅")

print(f"\n🌍 Coverage:")
regions   = set(j.region for j in jobs)
countries = set(j.country for j in jobs)
cities    = set(j.city for j in jobs)
print(f"  Regions   : {len(regions)}")
print(f"  Countries : {len(countries)}")
print(f"  Cities    : {len(cities)}")

print(f"\n💼 Job Categories:")
categories = {}
for job in jobs:
    categories[job.category] = categories.get(job.category, 0) + 1
for cat, count in sorted(categories.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat}: {count}")


  MILESTONE 3 - SYSTEM STATISTICS

📊 Data:
  Total jobs indexed : 644
  FAISS vectors      : 644
  LangChain Documents: 644

🔧 Stack:
  RAG Framework : LangChain
  Vector Store  : FAISS (Flat L2)
  Embeddings    : all-MiniLM-L6-v2
  LLM           : llama-3.3-70b-versatile via Groq
  Graph DB      : Neo4j (Milestone 2)
  Total Cost    : $0.00 ✅

🌍 Coverage:
  Regions   : 5
  Countries : 62
  Cities    : 233

💼 Job Categories:
  Data Scientist: 128
  Cloud: 115
  Business Analyst: 107
  Hr: 106
  Software Developer: 103
  Ui/Ux: 85


---
# 💾 PART 8: Save FAISS Index

In [37]:
# Save FAISS index for reuse (no need to rebuild next time)
vectorstore.save_local("faiss_index_langchain")

# Save job metadata
job_metadata = [
    {
        'job_id'             : j.job_id,
        'category'           : j.category,
        'city'               : j.city,
        'country'            : j.country,
        'workplace'          : j.workplace,
        'employment_type'    : j.employment_type,
        'priority_class'     : j.priority_class,
        'demand_score'       : j.demand_score,
        'department_category': j.department_category,
    }
    for j in jobs
]

with open('job_metadata.json', 'w') as f:
    json.dump(job_metadata, f, indent=2)

print("✅ Saved:")
print("  • faiss_index_langchain/ (vector index)")
print("  • job_metadata.json     (job details)")
print("\n💡 Next time load with:")
print("  vectorstore = FAISS.load_local('faiss_index_langchain', embeddings_model)")


✅ Saved:
  • faiss_index_langchain/ (vector index)
  • job_metadata.json     (job details)

💡 Next time load with:
  vectorstore = FAISS.load_local('faiss_index_langchain', embeddings_model)


---
# 📥 PART 9: Download Results

In [38]:
from google.colab import files
import shutil

# Zip the FAISS index folder
shutil.make_archive('faiss_index_langchain', 'zip', 'faiss_index_langchain')

files.download('faiss_index_langchain.zip')
files.download('job_metadata.json')

print("✅ Files downloaded!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Files downloaded!


---

# ✅ MILESTONE 3 COMPLETE!

## 🎯 What We Built

✅ **LangChain RAG Pipeline** — complete retrieval augmented generation  
✅ **FAISS Vector Store** — 644 job embeddings indexed and searchable  
✅ **SentenceTransformers** — local embeddings, no API key needed  
✅ **Groq (Llama 3)** — free LLM for answer generation  
✅ **Neo4j Integration** — loads directly from Milestone 2 graph  
✅ **Natural Language Search** — ask questions in plain English  

## 🔧 Tech Stack
| Component            |           Tool |
|---|---|
| RAG Framework | LangChain |
| Vector Store | FAISS |
| Embeddings | all-MiniLM-L6-v2 |
| LLM | Llama 3 via Groq |
| Graph DB | Neo4j |